### Imports

In [1]:
import os
import re

import numpy as np
import pandas as pd

from functools import *
from itertools import product

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

import umap
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import make_scorer, accuracy_score
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from umap import UMAP
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import warnings
warnings.filterwarnings("ignore")



### Helpers

In [2]:
def file_path(file_name):
    return os.path.join(os.getcwd(), file_name)


def load_data(path):

    data_frame = pd.read_csv(
            f"{path}" , 
            parse_dates = ["job_posted_date"], 
            date_format="%Y/%m"
            )
    
    return data_frame

def obj2cat(data_frame):

    for col in data_frame.select_dtypes(include = 'object'):
        data_frame[col] = data_frame[col].astype('category')

    return data_frame


def time_based_split(data_frame, date_field):

    data_frame = data_frame.sort_values(by=date_field).reset_index(drop=True)
    data_frame.drop("job_posted_date", axis=1, inplace=True)

    return data_frame

def add_date_element(data_frame, time_field):
    
    data_frame[time_field] = pd.to_datetime(data_frame[time_field], errors='coerce', infer_datetime_format=True)

    prefix = re.sub(r'[Dd]ate$', '', time_field)

    datetime_attrs = [
        'year', 'month',
        'is_month_end', 'is_month_start',
        'is_quarter_end', 'is_quarter_start',
        'is_year_end', 'is_year_start'
    ]

    for attr in datetime_attrs:
        new_col = f"{prefix}{attr.capitalize() if not attr.startswith('is_') else attr.title().replace('_', '')}"
        data_frame[new_col] = getattr(data_frame[time_field].dt, attr)

    return data_frame



def scaler(data_frame, scale_cols):
    
    scaler = StandardScaler()
    scaled_vals = scaler.fit_transform(data_frame[desc_cols])
    df_scaled = data_frame.copy()
    df_scaled[scale_cols] = scaled_vals

    return df_scaled




def apply_dim_reduction(data_frame_scaled, cols2reduce, n_components, method='pca',  merge_on="obs", drop_cols = False):
    
    if method == 'pca':
        model = PCA(n_components=n_components)
        prefix = 'pca'
    elif method == 'umap':
        model = umap.UMAP(n_components=n_components,  metric='cosine')
        prefix = 'umap'
    else:
        raise ValueError("Method must be either 'pca' or 'umap'.")
        
    reduced = model.fit_transform(data_frame_scaled[cols2reduce])
    reduced_df = pd.DataFrame(reduced, columns=[f'{prefix}_{i+1}' for i in range(reduced.shape[1])])
    reduced_df[merge_on] = data_frame_scaled[merge_on].values
    full_df = pd.merge(data_frame_scaled, reduced_df, on=merge_on)

    if drop_cols:
        full_df.drop(columns=cols2reduce, axis=1, inplace=True)

    return full_df
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD

def clean_and_apply_svd(
    df, 
    job_desc_cols, 
    n_components=10, 
    prefix="svd_topic_", 
    
    fillna=True
):
    df = df.copy()
    df["__obs__"] = df.index  # temporary ID for merging back later

    # 1. Extract the matrix
    X = df[job_desc_cols].values

    # 2. Filter out all-zero rows
    non_zero_mask = np.any(X != 0, axis=1)
    X_non_zero = X[non_zero_mask]
    df_non_zero = df[non_zero_mask].copy()

    # 3. Remove duplicate vectors
    seen = set()
    unique_indices = []

    for idx, row in enumerate(X_non_zero):
        row_tuple = tuple(np.round(row, decimals=6))  # float-safe deduplication
        if row_tuple not in seen:
            seen.add(row_tuple)
            unique_indices.append(idx)

    X_unique = X_non_zero[unique_indices]
    df_unique = df_non_zero.iloc[unique_indices].copy()

    # 4. Apply TruncatedSVD
    svd = TruncatedSVD(n_components=n_components)
    W = svd.fit_transform(X_unique)

    topic_cols = [f"{prefix}{i}" for i in range(n_components)]
    topic_df = pd.DataFrame(W, columns=topic_cols, index=df_unique.index)

    df_unique = pd.concat([df_unique, topic_df], axis=1)

    # 5. Merge topic features back to full DataFrame
    df_final = df.merge(
        df_unique[["__obs__"] + topic_cols],
        on="__obs__",
        how="left"
    ).drop(columns="__obs__")

    # 6. Fill NaNs (for removed rows) with 0 or leave as NaN
    if fillna:
        df_final[topic_cols] = df_final[topic_cols].fillna(0)

    return df_final




def plot_feature_importance(model, feature_names, top_n=20):
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    elif hasattr(model, 'get_score'):  # XGBoost specific
        importances_dict = model.get_score(importance_type='weight')
        importances = [importances_dict.get(f'f{i}', 0) for i in range(len(feature_names))]
    else:
        raise ValueError("Model does not provide feature importances.")

    # Build DataFrame
    feat_imp = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False).head(top_n)

    # Plot
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feat_imp, x='Importance', y='Feature')
    plt.title(f'Top {top_n} Feature Importances')
    plt.tight_layout()
    plt.show()

    return feat_imp



def preprocess_data(file_name):
    
    path_train = file_path(file_name)
    df_raw = load_data(path_train)
    df_raw["job_state"] = df_raw["job_state"].astype(str).fillna('missing')
    df_raw['state_missing'] = df_raw['job_state'] == "missing"
    df_raw = df_raw.applymap(lambda x: np.nan if pd.isna(x) else x) 
    df_raw = obj2cat(df_raw)
    df_raw = add_date_element(df_raw, "job_posted_date")

    return df_raw

In [200]:
import optuna
import shap
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.decomposition import PCA, TruncatedSVD, NMF, FactorAnalysis, FastICA, KernelPCA, SparsePCA, IncrementalPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from xgboost import XGBClassifier, XGBRegressor
from umap import UMAP

def evaluate_with_shap_and_top_features(study, X, y, desc_cols, model_type='classifier', top_n=10, cv_folds=None):
    """
    Evaluate best parameters with SHAP analysis, matching exact training pipeline
    """
    # 1. Get best parameters
    best_params = study.best_trial.params.copy()
    print("Best Optuna params:")
    for k,v in best_params.items():
        print(f"  {k}: {v}")
    print()
    
    # 2. Apply the EXACT same preprocessing as  training pipeline
    print("Applying dimensionality reduction...")
    X_desc = X[desc_cols].copy()
    
    # Apply dimensionality reduction with best parameters 
    try:
        if best_params['reduction_method'] == 'pca':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = PCA(n_components=best_params['n_components'])
            X_reduced = reducer.fit_transform(X_desc_processed)
        
        elif best_params['reduction_method'] == 'umap':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = UMAP(
                n_components=best_params['n_components'], 
                n_neighbors=best_params.get('n_neighbors', 15),
                min_dist=best_params.get('min_dist', 0.1)
            )
            X_reduced = reducer.fit_transform(X_desc_processed)
        
        elif best_params['reduction_method'] == 'truncated_svd':
            X_desc_processed = X_desc.values if hasattr(X_desc, 'values') else X_desc
            reducer = TruncatedSVD(n_components=best_params['n_components'])
            X_reduced = reducer.fit_transform(X_desc_processed)
            
        elif best_params['reduction_method'] == 'nmf':
            scaler = MinMaxScaler()
            X_desc_processed = scaler.fit_transform(X_desc) + 1e-8
            reducer = NMF(
                n_components=best_params['n_components'], 
                init=best_params.get('nmf_init', 'random'),
               
                max_iter=400
            )
            X_reduced = reducer.fit_transform(X_desc_processed)
            
        elif best_params['reduction_method'] == 'factor_analysis':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = FactorAnalysis(n_components=best_params['n_components'])
            X_reduced = reducer.fit_transform(X_desc_processed)
            
        elif best_params['reduction_method'] == 'ica':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = FastICA(
                n_components=best_params['n_components'],
                algorithm=best_params.get('ica_algorithm', 'parallel'),
                fun=best_params.get('ica_fun', 'logcosh'),
                
                max_iter=400
            )
            X_reduced = reducer.fit_transform(X_desc_processed)
            
        elif best_params['reduction_method'] == 'kernel_pca':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = KernelPCA(
                n_components=best_params['n_components'],
                kernel=best_params.get('kernel', 'linear')
            )
            X_reduced = reducer.fit_transform(X_desc_processed)
            
        elif best_params['reduction_method'] == 'sparse_pca':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = SparsePCA(
                n_components=best_params['n_components'],
               
                max_iter=400
            )
            X_reduced = reducer.fit_transform(X_desc_processed)
            
        elif best_params['reduction_method'] == 'lda':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            y_for_lda = LabelEncoder().fit_transform(y) if y.dtype == object else y
            n_classes = len(np.unique(y_for_lda))
            n_components_lda = min(best_params['n_components'], n_classes - 1, X_desc_processed.shape[1])
            
            if n_components_lda <= 0:
                reducer = PCA(n_components=best_params['n_components'])
                X_reduced = reducer.fit_transform(X_desc_processed)
            else:
                reducer = LinearDiscriminantAnalysis(
                    n_components=n_components_lda,
                    solver=best_params.get('lda_solver', 'svd')
                )
                X_reduced = reducer.fit_transform(X_desc_processed, y_for_lda)
                
        elif best_params['reduction_method'] == 'robust_pca':
            X_desc_processed = RobustScaler().fit_transform(X_desc)
            reducer = PCA(n_components=best_params['n_components'])
            X_reduced = reducer.fit_transform(X_desc_processed)
            
        elif best_params['reduction_method'] == 'incremental_pca':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = IncrementalPCA(n_components=best_params['n_components'])
            X_reduced = reducer.fit_transform(X_desc_processed)
            
        else:
            # Fallback to PCA for any other methods
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = PCA(n_components=best_params['n_components'])
            X_reduced = reducer.fit_transform(X_desc_processed)
    
    except Exception as e:
        print(f"Warning: {best_params['reduction_method']} failed, using PCA. Error: {str(e)}")
        X_desc_processed = StandardScaler().fit_transform(X_desc)
        reducer = PCA(n_components=best_params['n_components'])
        X_reduced = reducer.fit_transform(X_desc_processed)
    
    # 3. Create final dataset 
    X_other = X.drop(columns=desc_cols)
    X_full = pd.concat([
        X_other.reset_index(drop=True), 
        pd.DataFrame(X_reduced, columns=[f"{best_params['reduction_method']}_{i}" for i in range(X_reduced.shape[1])])
    ], axis=1)
    
    print(f"Final dataset shape: {X_full.shape}")
    print(f"Reduced components: {X_reduced.shape[1]}")
    print(f"Other features: {X_other.shape[1]}")
    
    # 4. Prepare target variable
    if model_type == 'classifier':
        y_encoded = LabelEncoder().fit_transform(y) if y.dtype == object else y
    else:
        y_encoded = y
    
    # 5. Create model with best parameters 
    xgb_params = {k: v for k, v in best_params.items() if k not in [
        'n_components', 'n_splits', 'reduction_method', 'kernel', 
        'lda_solver', 'n_neighbors', 'min_dist', 'nmf_init', 
        'ica_algorithm', 'ica_fun', 'encoding_dim', 'epochs'
    ]}
    
    if model_type == 'classifier':
        model = XGBClassifier(
            **xgb_params,
            use_label_encoder=False, 
            verbosity=0, 
            enable_categorical=True
        )
        cv = StratifiedKFold(n_splits=best_params['n_splits'], shuffle=True)
        scoring = 'accuracy'
    else:
        model = XGBRegressor(
            **xgb_params,
            enable_categorical=True
        )
        cv = KFold(n_splits=best_params['n_splits'], shuffle=True)
        scoring = 'neg_root_mean_squared_error'
    
    # 6. Train model and evaluate
    model.fit(X_full, y_encoded)
    train_score = model.score(X_full, y_encoded)
    cv_scores = cross_val_score(model, X_full, y_encoded, cv=cv, scoring=scoring)
    cv_mean, cv_std = cv_scores.mean(), cv_scores.std()
    
    print(f"Full features → Train score: {train_score:.4f}, CV score: {cv_mean:.4f} ± {cv_std:.4f}")
    
    # 7. Compute SHAP values - FIXED VERSION
    try:
        print("Computing SHAP values...")
        
        # Use TreeExplainer for XGBoost models
        explainer = shap.TreeExplainer(model)
        
        # Sample data if dataset is large (SHAP can be slow)
        if len(X_full) > 1000:
            print(f"Large dataset ({len(X_full)} rows), sampling 1000 rows for SHAP calculation...")
            sample_idx = np.random.choice(len(X_full), size=1000, replace=False)
            X_sample = X_full.iloc[sample_idx]
        else:
            X_sample = X_full
        
        shap_vals = explainer.shap_values(X_sample)
        
        # Handle different SHAP output formats - FIXED
        if isinstance(shap_vals, list):  # Multi-class classification
            print(f"Multi-class detected: {len(shap_vals)} classes")
            # For multi-class, take mean absolute SHAP values across all classes
            shap_vals_stacked = np.stack(shap_vals, axis=2)  # Shape: (samples, features, classes)
            mean_abs_shap = np.abs(shap_vals_stacked).mean(axis=(0, 2))  # Mean across samples and classes
        elif len(shap_vals.shape) == 3:  # Another multi-class format
            print(f"Multi-class detected: shape {shap_vals.shape}")
            mean_abs_shap = np.abs(shap_vals).mean(axis=(0, 2))  # Mean across samples and classes
        else:  # Binary classification or regression
            print("Binary classification or regression detected")
            mean_abs_shap = np.abs(shap_vals).mean(axis=0)
        
        # Ensure we have the right number of features
        if len(mean_abs_shap) != X_full.shape[1]:
            print(f"Warning: SHAP values length ({len(mean_abs_shap)}) doesn't match features ({X_full.shape[1]})")
            raise ValueError("SHAP values dimension mismatch")
        
        feature_importance = pd.Series(mean_abs_shap, index=X_full.columns)
        top_feats = feature_importance.sort_values(ascending=False).head(top_n).index.tolist()
        
        print(f"\nTop {top_n} features by mean |SHAP|:")
        for i, feat in enumerate(top_feats, 1):
            print(f"  {i}. {feat}: {feature_importance[feat]:.4f}")
        
        # Separate reduced features from other features
        reduced_feats = [f for f in top_feats if f.startswith(best_params['reduction_method'] + '_')]
        other_feats = [f for f in top_feats if not f.startswith(best_params['reduction_method'] + '_')]
        
        print(f"\nBreakdown:")
        print(f"  Reduced features ({best_params['reduction_method']}): {len(reduced_feats)}")
        print(f"  Other features: {len(other_feats)}")
        
    except Exception as e:
        print(f"SHAP calculation failed: {e}")
        print(f"SHAP values type: {type(shap_vals) if 'shap_vals' in locals() else 'Not computed'}")
        if 'shap_vals' in locals():
            if isinstance(shap_vals, list):
                print(f"SHAP values list length: {len(shap_vals)}")
                print(f"First element shape: {shap_vals[0].shape if len(shap_vals) > 0 else 'Empty'}")
            else:
                print(f"SHAP values shape: {shap_vals.shape}")
        
        print("Using XGBoost feature importance instead...")
        
        # Fallback to XGBoost feature importance
        feature_importance = pd.Series(model.feature_importances_, index=X_full.columns)
        top_feats = feature_importance.sort_values(ascending=False).head(top_n).index.tolist()
        
        print(f"Top {top_n} features by XGBoost importance:")
        for i, feat in enumerate(top_feats, 1):
            print(f"  {i}. {feat}: {feature_importance[feat]:.4f}")
    
    # 8. Retrain with only top features
    X_reduced_top = X_full[top_feats]
    
    if model_type == 'classifier':
        model2 = XGBClassifier(
            **xgb_params,
            use_label_encoder=False, 
            verbosity=0, 
            enable_categorical=True
        )
    else:
        model2 = XGBRegressor(
            **xgb_params,
            enable_categorical=True
        )
    
    model2.fit(X_reduced_top, y_encoded)
    train2 = model2.score(X_reduced_top, y_encoded)
    cv2_scores = cross_val_score(model2, X_reduced_top, y_encoded, cv=cv, scoring=scoring)
    cv2 = cv2_scores.mean()
    cv2_std = cv2_scores.std()
    
    print(f"\nTop {top_n} features → Train score: {train2:.4f}, CV score: {cv2:.4f} ± {cv2_std:.4f}")
    
    return {
        'best_params': best_params,
        'top_features': top_feats,
        'scores_full': (train_score, cv_mean, cv_std),
        'scores_reduced': (train2, cv2, cv2_std),
        'feature_importance': feature_importance,
        'final_dataset_shape': X_full.shape,
        'reduction_method': best_params['reduction_method'],
        'n_components': best_params['n_components']
    }

# Usage function that matches  training setup
def run_shap_evaluation(study, X, y, desc_cols, model_type="classifier", top_n=10, cv_folds=None):
    """
    Run SHAP evaluation matching exact training pipeline
    
    Parameters:
    - study:  completed Optuna study
    - X:  feature DataFrame (same as used in training)
    - y:  target variable (same as used in training)  
    - desc_cols: List of description columns (same as used in training)
    - model_type: 'classifier' or 'regressor'
    - top_n: Number of top features to analyze
    """
    
    # Get cv_folds from best_params
    cv_folds = study.best_trial.params.get('n_splits', cv_folds)
    
    print(f"Running SHAP analysis with:")
    print(f"  Model type: {model_type}")
    print(f"  Top N features: {top_n}")
    print(f"  CV folds: {cv_folds}")
    print(f"  Description columns: {len(desc_cols)}")
    print(f"  Total features: {X.shape[1]}")
    print("-" * 50)
    
    results = evaluate_with_shap_and_top_features(
        study, X, y, desc_cols,
        model_type=model_type,
        top_n=top_n,
        cv_folds=cv_folds
    )
    
    return results



### Pre Processing

In [3]:
df_raw = preprocess_data("train.csv")


job_counts = df_raw['job_title'].value_counts()

# Step 2: Create a mapping based on count ranges
def categorize_job(job):
    count = job_counts.get(job, 0)
    if count < 25:
        return 'rare2'
    elif 25 <= count <= 55:
        return 'rare1'
    else:
        return job  # keep the original category

# Step 3: Apply mapping to the column
df_raw['job_title'] = df_raw['job_title'].apply(categorize_job)

# Calculate value counts
state_counts = df_raw['job_state'].value_counts()

# Define threshold
threshold = 30

# Identify states to keep
states_to_keep = state_counts[state_counts >= threshold].index

# Apply mapping
df_raw['job_state'] = df_raw['job_state'].apply(
    lambda x: x if pd.isna(x) or x in states_to_keep else 'other'
)

df_raw['job_title'] = df_raw['job_title'].astype('category')
df_raw['job_state'] = df_raw['job_state'].astype('category')



desc_cols = [col for col in df_raw.columns if col.startswith("job_desc_")]
feature_cols = [col for col in df_raw.columns if col.startswith("feature_")]
df_raw = time_based_split(df_raw, "job_posted_date")


df_raw['salary_cat_num'] = df_raw['salary_category'].map({'Low': 0, 'Medium': 1, 'High': 2})
y = df_raw['salary_cat_num'].values
X = df_raw.drop(['obs','salary_category'], axis=1)
y

le = LabelEncoder()
y = le.fit_transform(y);
y

array([1, 2, 0, ..., 2, 0, 0], dtype=int64)

In [4]:
cat_col = X.select_dtypes(include='category').columns.tolist()

In [5]:
X.drop(['salary_cat_num'], axis=1, inplace=True)

# Trainer

In [ ]:
import optuna
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler, RobustScaler
from sklearn.decomposition import PCA, TruncatedSVD, NMF, FactorAnalysis, FastICA, KernelPCA, SparsePCA, IncrementalPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neural_network import MLPRegressor
from xgboost import XGBClassifier, XGBRegressor
from umap import UMAP

#  original objective function
def objective(trial, X, y, desc_cols, model_type):
    # Parameters to tune
    params = {
        'max_depth': trial.suggest_int('max_depth', 3,6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.13),
        'n_estimators': trial.suggest_int('n_estimators', 100, 250),
        'subsample': trial.suggest_float('subsample', 0.8, 0.8),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 4, 10),
        'gamma': trial.suggest_int('gamma', 2, 10),
        'n_components': trial.suggest_int('n_components', 5, 50),
        'n_splits': trial.suggest_int('n_splits', 4, 6),
        'reduction_method': trial.suggest_categorical('reduction_method', [
                'incremental_pca', 'factor_analysis',  'truncated_svd', 'ica', 'kernel_pca', 'umap', 'pca', 'nmf', 'robust_pca'
        ]),
    }
    
            
            
    # Additional parameters for specific methods
    if params['reduction_method'] == 'kernel_pca':
        params['kernel'] = trial.suggest_categorical('kernel', ['linear', 'poly', 'rbf', 'sigmoid'])
    elif params['reduction_method'] == 'lda':
        params['solver'] = trial.suggest_categorical('lda_solver', ['svd', 'lsqr', 'eigen'])
    elif params['reduction_method'] == 'umap':
        params['n_neighbors'] = trial.suggest_int('n_neighbors', 5, 50)
        params['min_dist'] = trial.suggest_float('min_dist', 0.01, 0.5)
    elif params['reduction_method'] == 'nmf':
        params['init'] = trial.suggest_categorical('nmf_init', ['random', 'nndsvd', 'nndsvda'])
    elif params['reduction_method'] == 'ica':
        params['algorithm'] = trial.suggest_categorical('ica_algorithm', ['parallel', 'deflation'])
        params['fun'] = trial.suggest_categorical('ica_fun', ['logcosh', 'exp', 'cube'])
    elif params['reduction_method'] == 'autoencoder':
        params['encoding_dim'] = trial.suggest_int('encoding_dim', 32, 128)
        params['epochs'] = trial.suggest_int('epochs', 50, 150)
    
    # Extract only description columns to reduce
    X_desc = X[desc_cols].copy()
    
    # Apply selected dimensionality reduction with appropriate preprocessing
    try:
        if params['reduction_method'] == 'pca':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = PCA(n_components=params['n_components'])
        
        elif params['reduction_method'] == 'umap':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = UMAP(
                n_components=params['n_components'], 
                n_neighbors=params['n_neighbors'],
                min_dist=params['min_dist']
            )
        
        elif params['reduction_method'] == 'truncated_svd':
            # TruncatedSVD works better without centering (standardization centers data)
            X_desc_processed = X_desc.values if hasattr(X_desc, 'values') else X_desc
            reducer = TruncatedSVD(n_components=params['n_components'])
        
        elif params['reduction_method'] == 'nmf':
            # NMF requires strictly non-negative data
            # Use MinMaxScaler to ensure [0,1] range, then add small epsilon
            scaler = MinMaxScaler()
            X_desc_processed = scaler.fit_transform(X_desc) + 1e-8
            reducer = NMF(
                n_components=params['n_components'], 
                init=params['init'],
                
                max_iter=400
            )
            X_reduced = reducer.fit_transform(X_desc_processed)
        
        elif params['reduction_method'] == 'factor_analysis':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = FactorAnalysis(n_components=params['n_components'])
        
        elif params['reduction_method'] == 'ica':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = FastICA(
                n_components=params['n_components'],
                algorithm=params['algorithm'],
                fun=params['fun'],
                
                max_iter=400
            )
        
        elif params['reduction_method'] == 'kernel_pca':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = KernelPCA(
                n_components=params['n_components'],
                kernel=params['kernel']
            )
        
        elif params['reduction_method'] == 'sparse_pca':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = SparsePCA(
                n_components=params['n_components'],
           
                max_iter=400
            )
        
        elif params['reduction_method'] == 'lda':
            # LDA works better with standardized data
            X_desc_processed = StandardScaler().fit_transform(X_desc)
        
        elif params['reduction_method'] == 'robust_pca':
            # Robust PCA handles outliers and negative values well
            X_desc_processed = RobustScaler().fit_transform(X_desc)
            reducer = PCA(n_components=params['n_components'])
        
        elif params['reduction_method'] == 'incremental_pca':
            # Good for large datasets, handles negative values
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = IncrementalPCA(n_components=params['n_components'])
        
        
        
        # Apply transformation (except for NMF, LDA, and autoencoder which were already processed)
        if params['reduction_method'] not in ['nmf', 'lda', 'autoencoder']:
            X_reduced = reducer.fit_transform(X_desc_processed)
        elif params['reduction_method'] == 'lda':
            # LDA needs target variable for supervised dimensionality reduction
            y_for_lda = LabelEncoder().fit_transform(y) if y.dtype == object else y
            n_classes = len(np.unique(y_for_lda))
            n_components_lda = min(params['n_components'], n_classes - 1, X_desc_processed.shape[1])
            
            if n_components_lda <= 0:
                # Fallback to PCA if LDA is not applicable
                reducer = PCA(n_components=params['n_components'])
                X_reduced = reducer.fit_transform(X_desc_processed)
            else:
                reducer = LinearDiscriminantAnalysis(
                    n_components=n_components_lda,
                    solver=params['solver']
                )
                X_reduced = reducer.fit_transform(X_desc_processed, y_for_lda)
    
    except Exception as e:
        # Fallback to PCA if any method fails
        print(f"Warning: {params['reduction_method']} failed, falling back to PCA. Error: {str(e)}")
        X_desc_processed = StandardScaler().fit_transform(X_desc)
        reducer = PCA(n_components=params['n_components'])
        X_reduced = reducer.fit_transform(X_desc_processed)
    
    # Create final X with reduced + rest
    X_other = X.drop(columns=desc_cols)
    df_svd = pd.concat([
        X_other.reset_index(drop=True), 
        pd.DataFrame(X_reduced, columns=[f"{params['reduction_method']}_{i}" for i in range(X_reduced.shape[1])])
    ], axis=1)
    
    # Prepare model and CV
    if model_type == 'classifier':
        y_encoded = LabelEncoder().fit_transform(y) if y.dtype == object else y
        model = XGBClassifier(
            **{k: v for k, v in params.items() if k not in [
                'n_components', 'n_splits', 'reduction_method', 'kernel', 
                'lda_solver', 'n_neighbors', 'min_dist', 'nmf_init', 
                'ica_algorithm', 'ica_fun', 'encoding_dim', 'epochs'
            ]},
            use_label_encoder=False, verbosity=0, enable_categorical=True
        )
        cv = StratifiedKFold(n_splits=params['n_splits'], shuffle=True)
        scoring = 'accuracy'
    else:
        y_encoded = y
        model = XGBRegressor(
            **{k: v for k, v in params.items() if k not in [
                'n_components', 'n_splits', 'reduction_method', 'kernel',
                'lda_solver', 'n_neighbors', 'min_dist', 'nmf_init',
                'ica_algorithm', 'ica_fun', 'encoding_dim', 'epochs'
            ]},
            enable_categorical=True
        )
        cv = KFold(n_splits=params['n_splits'], shuffle=True)
        scoring = 'neg_root_mean_squared_error'
    
    # Cross-validation
    scores = cross_val_score(model, df_svd, y_encoded, cv=cv, scoring=scoring)
    return scores.mean() if model_type == 'classifier' else -scores.mean()

def evaluate_best_params(best_params, X, y, desc_cols, model_type):
    """Evaluate the best parameters and return train and CV accuracy"""
    
    # Apply the same preprocessing as in objective function
    X_desc = X[desc_cols].copy()
    
    # Apply dimensionality reduction with best parameters
    try:
        if best_params['reduction_method'] == 'pca':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = PCA(n_components=best_params['n_components'])
            X_reduced = reducer.fit_transform(X_desc_processed)
        
        elif best_params['reduction_method'] == 'umap':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = UMAP(
                n_components=best_params['n_components'], 
                n_neighbors=best_params.get('n_neighbors', 15),
                min_dist=best_params.get('min_dist', 0.1)
            )
            X_reduced = reducer.fit_transform(X_desc_processed)
        
        elif best_params['reduction_method'] == 'truncated_svd':
            X_desc_processed = X_desc.values if hasattr(X_desc, 'values') else X_desc
            reducer = TruncatedSVD(n_components=best_params['n_components'])
            X_reduced = reducer.fit_transform(X_desc_processed)
            
        elif best_params['reduction_method'] == 'nmf':
            scaler = MinMaxScaler()
            X_desc_processed = scaler.fit_transform(X_desc) + 1e-8
            reducer = NMF(
                n_components=best_params['n_components'], 
                init=best_params.get('init', 'random'),
              
                max_iter=400
            )
            X_reduced = reducer.fit_transform(X_desc_processed)
            
        elif best_params['reduction_method'] == 'factor_analysis':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = FactorAnalysis(n_components=best_params['n_components'])
            X_reduced = reducer.fit_transform(X_desc_processed)
            
        elif best_params['reduction_method'] == 'ica':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = FastICA(
                n_components=best_params['n_components'],
                algorithm=best_params.get('algorithm', 'parallel'),
                fun=best_params.get('fun', 'logcosh'),
                
                max_iter=400
            )
            X_reduced = reducer.fit_transform(X_desc_processed)
            
        elif best_params['reduction_method'] == 'kernel_pca':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = KernelPCA(
                n_components=best_params['n_components'],
                kernel=best_params.get('kernel', 'linear')
            )
            X_reduced = reducer.fit_transform(X_desc_processed)
            
        elif best_params['reduction_method'] == 'sparse_pca':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = SparsePCA(
                n_components=best_params['n_components'],
             
                max_iter=400
            )
            X_reduced = reducer.fit_transform(X_desc_processed)
            
        elif best_params['reduction_method'] == 'lda':
            # Use whatever  training data variable is called
            scaler = StandardScaler()
            X_desc_processed = scaler.fit_transform(X_desc)  #  current training data variable
    
            # But make sure X_desc is ONLY training data, not full dataset!
            y_for_lda = LabelEncoder().fit_transform(y) if y.dtype == object else y  # Only training labels
    
            n_classes = len(np.unique(y_for_lda))
            n_components_lda = min(best_params['n_components'], n_classes - 1, X_desc_processed.shape[1])
    
            if n_components_lda <= 0:
                reducer = PCA(n_components=best_params['n_components'])
                X_reduced = reducer.fit_transform(X_desc_processed)
            else:
                reducer = LinearDiscriminantAnalysis(
                    n_components=n_components_lda,
                    solver=best_params.get('solver', 'svd')
                )
                X_reduced = reducer.fit_transform(X_desc_processed, y_for_lda)             
        elif best_params['reduction_method'] == 'robust_pca':
                X_desc_processed = RobustScaler().fit_transform(X_desc)
                reducer = PCA(n_components=best_params['n_components'])
                X_reduced = reducer.fit_transform(X_desc_processed)
        
        elif best_params['reduction_method'] == 'incremental_pca':
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = IncrementalPCA(n_components=best_params['n_components'])
            X_reduced = reducer.fit_transform(X_desc_processed)
            
        else:
            # Fallback to PCA for any other methods
            X_desc_processed = StandardScaler().fit_transform(X_desc)
            reducer = PCA(n_components=best_params['n_components'])
            X_reduced = reducer.fit_transform(X_desc_processed)
    
    except Exception as e:
        print(f"Warning: {best_params['reduction_method']} failed, using PCA. Error: {str(e)}")
        X_desc_processed = StandardScaler().fit_transform(X_desc)
        reducer = PCA(n_components=best_params['n_components'])
        X_reduced = reducer.fit_transform(X_desc_processed)
    
    # Create final dataset
    X_other = X.drop(columns=desc_cols)
    df_final = pd.concat([
        X_other.reset_index(drop=True), 
        pd.DataFrame(X_reduced, columns=[f"{best_params['reduction_method']}_{i}" for i in range(X_reduced.shape[1])])
    ], axis=1)
    
    # Prepare model
    if model_type == 'classifier':
        y_encoded = LabelEncoder().fit_transform(y) if y.dtype == object else y
        model = XGBClassifier(
            **{k: v for k, v in best_params.items() if k not in [
                'n_components', 'n_splits', 'reduction_method', 'kernel', 
                'lda_solver', 'n_neighbors', 'min_dist', 'nmf_init', 
                'ica_algorithm', 'ica_fun', 'encoding_dim', 'epochs'
            ]},
            use_label_encoder=False, verbosity=0, enable_categorical=True
        )
        cv = StratifiedKFold(n_splits=best_params['n_splits'], shuffle=True)
        scoring = 'accuracy'
    else:
        y_encoded = y
        model = XGBRegressor(
            **{k: v for k, v in best_params.items() if k not in [
                'n_components', 'n_splits', 'reduction_method', 'kernel',
                'lda_solver', 'n_neighbors', 'min_dist', 'nmf_init',
                'ica_algorithm', 'ica_fun', 'encoding_dim', 'epochs'
            ]},
            enable_categorical=True
        )
        cv = KFold(n_splits=best_params['n_splits'], shuffle=True)
        scoring = 'neg_root_mean_squared_error'
    
    # Calculate train accuracy
    model.fit(df_final, y_encoded)
    train_score = model.score(df_final, y_encoded)
    
    # Calculate CV accuracy
    cv_scores = cross_val_score(model, df_final, y_encoded, cv=cv, scoring=scoring)
    cv_score = cv_scores.mean()
    
    return train_score, cv_score, cv_scores.std()







# Set  parameters here
model_type = "classifier"  # or "regressor"

# Uncomment and set  data:
# X = feature_dataframe
# y = target_variable
# desc_cols = description_columns_list

print("Starting hyperparameter optimization...")
print(f"Model type: {model_type}")
print(f"Number of trials: 150")
print("-" * 50)

# Create and run study
study = optuna.create_study(direction="maximize" if model_type == "classifier" else "minimize")
study.optimize(lambda trial: objective(trial, X, y, desc_cols, model_type), n_trials=500)

# Get best parameters
best_params = study.best_trial.params

best_value = study.best_value

print("\n" + "="*50)
print("OPTIMIZATION RESULTS")
print("="*50)
print(f"Best trial value: {best_value:.6f}")
print(f"Best parameters:")
for key, value in best_params.items():
    print(f"  {key}: {value}")

# Evaluate best parameters
print("\n" + "="*50)
print("DETAILED EVALUATION")
print("="*50)

train_acc, cv_acc, cv_std = evaluate_best_params(best_params, X, y, desc_cols, model_type)

if model_type == "classifier":
    print(f"Train Accuracy: {train_acc:.6f}")
    print(f"CV Accuracy: {cv_acc:.6f} (±{cv_std:.6f})")
else:
    print(f"Train RMSE: {train_acc:.6f}")
    print(f"CV RMSE: {cv_acc:.6f} (±{cv_std:.6f})")

print(f"Dimensionality Reduction: {best_params['reduction_method']}")
print(f"Number of Components: {best_params['n_components']}")
print(f"CV Folds: {best_params['n_splits']}")

[I 2025-06-15 07:24:35,546] A new study created in memory with name: no-name-73b4ab4b-9362-4ce1-890e-00b27e6653ee


Starting hyperparameter optimization...
Model type: classifier
Number of trials: 150
--------------------------------------------------


[I 2025-06-15 07:24:37,251] Trial 0 finished with value: 0.6242230120076638 and parameters: {'max_depth': 5, 'learning_rate': 0.028356484783701744, 'n_estimators': 147, 'subsample': 0.8, 'colsample_bytree': 0.7975481045963058, 'reg_alpha': 7.668004325322113, 'reg_lambda': 4.14566111303049, 'min_child_weight': 10, 'gamma': 8, 'n_components': 12, 'n_splits': 6, 'reduction_method': 'nmf', 'nmf_init': 'nndsvd'}. Best is trial 0 with value: 0.6242230120076638.
[I 2025-06-15 07:24:38,479] Trial 1 finished with value: 0.6640742690828251 and parameters: {'max_depth': 3, 'learning_rate': 0.10818574187002918, 'n_estimators': 133, 'subsample': 0.8, 'colsample_bytree': 0.6113428912499118, 'reg_alpha': 8.465196349063746, 'reg_lambda': 8.405074081123757, 'min_child_weight': 9, 'gamma': 4, 'n_components': 17, 'n_splits': 6, 'reduction_method': 'factor_analysis'}. Best is trial 1 with value: 0.6640742690828251.
[I 2025-06-15 07:24:41,228] Trial 2 finished with value: 0.6953329530662687 and parameters:

In [216]:
results = run_shap_evaluation(
    study=study,           #  completed Optuna study
    X=X,                   #  feature DataFrame  
    y=y,                   #  target variable
    desc_cols=desc_cols,   #  description columns list
    model_type="classifier", # or "regressor"
    top_n=20, cv_folds = best_params["n_splits"]
)



Running SHAP analysis with:
  Model type: classifier
  Top N features: 20
  CV folds: 4
  Description columns: 300
  Total features: 323
--------------------------------------------------
Best Optuna params:
  max_depth: 4
  learning_rate: 0.12455413371612384
  n_estimators: 212
  subsample: 0.801486676101801
  colsample_bytree: 0.7624412904398317
  reg_alpha: 1.290427677453912
  reg_lambda: 4.410219686600314
  min_child_weight: 4
  gamma: 2
  n_components: 48
  n_splits: 4
  reduction_method: kernel_pca
  kernel: linear

Applying dimensionality reduction...
Final dataset shape: (1280, 71)
Reduced components: 48
Other features: 23
Full features → Train score: 0.8711, CV score: 0.7266 ± 0.0239
Computing SHAP values...
Large dataset (1280 rows), sampling 1000 rows for SHAP calculation...
Multi-class detected: shape (1000, 71, 3)

Top 20 features by mean |SHAP|:
  1. feature_2: 0.4649
  2. kernel_pca_1: 0.1983
  3. feature_10: 0.1499
  4. kernel_pca_4: 0.1213
  5. feature_1: 0.0785
  6. f

In [217]:
def predict_on_test(best_params, X_train, y_train, df_raw_test, desc_cols, model_type='classifier'):
    from sklearn.pipeline import Pipeline
    
    # -------- Preprocessing: Fit on Train, Transform Both Train and Test --------
    # Process descriptive columns
    X_desc_train = X_train[desc_cols].copy()
    X_desc_test = df_raw_test[desc_cols].copy()

    # Apply corresponding preprocessing and dimensionality reduction
    try:
        if best_params['reduction_method'] == 'pca':
            scaler = StandardScaler()
            reducer = PCA(n_components=best_params['n_components'], random_state=42)

        elif best_params['reduction_method'] == 'umap':
            scaler = StandardScaler()
            reducer = UMAP(
                n_components=best_params['n_components'],
                n_neighbors=best_params.get('n_neighbors', 15),
                min_dist=best_params.get('min_dist', 0.1),
                random_state=42
            )

        elif best_params['reduction_method'] == 'truncated_svd':
            scaler = None  # no scaling for SVD
            reducer = TruncatedSVD(n_components=best_params['n_components'], random_state=42)

        elif best_params['reduction_method'] == 'nmf':
            scaler = MinMaxScaler()
            reducer = NMF(
                n_components=best_params['n_components'], 
                init=best_params.get('init', 'random'),
                random_state=42,
                max_iter=400
            )

        elif best_params['reduction_method'] == 'factor_analysis':
            scaler = StandardScaler()
            reducer = FactorAnalysis(n_components=best_params['n_components'], random_state=42)

        elif best_params['reduction_method'] == 'ica':
            scaler = StandardScaler()
            reducer = FastICA(
                n_components=best_params['n_components'],
                algorithm=best_params.get('algorithm', 'parallel'),
                fun=best_params.get('fun', 'logcosh'),
                random_state=42,
                max_iter=400
            )

        elif best_params['reduction_method'] == 'kernel_pca':
            scaler = StandardScaler()
            reducer = KernelPCA(
                n_components=best_params['n_components'],
                kernel=best_params.get('kernel', 'linear')
            )

        elif best_params['reduction_method'] == 'sparse_pca':
            scaler = StandardScaler()
            reducer = SparsePCA(n_components=best_params['n_components'])

        elif best_params['reduction_method'] == 'lda':
            scaler = StandardScaler()
            y_encoded = LabelEncoder().fit_transform(y_train) if y_train.dtype == object else y_train
            n_classes = len(np.unique(y_encoded))
            n_components_lda = min(best_params['n_components'], n_classes - 1, X_desc_train.shape[1])
            if n_components_lda <= 0:
                reducer = PCA(n_components=best_params['n_components'])
            else:
                reducer = LinearDiscriminantAnalysis(
                    n_components=n_components_lda,
                    solver=best_params.get('solver', 'svd')
                )
                reducer.fit(scaler.fit_transform(X_desc_train), y_encoded)
                X_train_reduced = reducer.transform(scaler.transform(X_desc_train))
                X_test_reduced = reducer.transform(scaler.transform(X_desc_test))

        elif best_params['reduction_method'] == 'robust_pca':
            scaler = RobustScaler()
            reducer = PCA(n_components=best_params['n_components'])

        elif best_params['reduction_method'] == 'incremental_pca':
            scaler = StandardScaler()
            reducer = IncrementalPCA(n_components=best_params['n_components'])

        # Fit and transform train, then transform test
        if best_params['reduction_method'] != 'lda':
            if scaler is not None:
                X_desc_train_scaled = scaler.fit_transform(X_desc_train)
                X_desc_test_scaled = scaler.transform(X_desc_test)
            else:
                X_desc_train_scaled = X_desc_train.values
                X_desc_test_scaled = X_desc_test.values

            X_train_reduced = reducer.fit_transform(X_desc_train_scaled)
            X_test_reduced = reducer.transform(X_desc_test_scaled)

    except Exception as e:
        print(f"Failed during dimensionality reduction: {e}")
        return None

    # Merge reduced with other features
    X_train_other = X_train.drop(columns=desc_cols).reset_index(drop=True)
    X_test_other = df_raw_test.drop(columns=desc_cols).reset_index(drop=True)
    X_train_final = pd.concat([
        X_train_other,
        pd.DataFrame(X_train_reduced, columns=[f"{best_params['reduction_method']}_{i}" for i in range(X_train_reduced.shape[1])])
    ], axis=1)
    X_test_final = pd.concat([
        X_test_other,
        pd.DataFrame(X_test_reduced, columns=[f"{best_params['reduction_method']}_{i}" for i in range(X_test_reduced.shape[1])])
    ], axis=1)

    # Prepare and train the model
    if model_type == 'classifier':
        y_encoded = LabelEncoder().fit_transform(y_train) if y_train.dtype == object else y_train
        model = XGBClassifier(
            **{k: v for k, v in best_params.items() if k not in [
                'n_components', 'n_splits', 'reduction_method', 'kernel',
                'lda_solver', 'n_neighbors', 'min_dist', 'nmf_init',
                'ica_algorithm', 'ica_fun', 'encoding_dim', 'epochs'
            ]},
            use_label_encoder=False, verbosity=0, enable_categorical=True
        )
    else:
        y_encoded = y_train
        model = XGBRegressor(
            **{k: v for k, v in best_params.items() if k not in [
                'n_components', 'n_splits', 'reduction_method', 'kernel',
                'lda_solver', 'n_neighbors', 'min_dist', 'nmf_init',
                'ica_algorithm', 'ica_fun', 'encoding_dim', 'epochs'
            ]},
            enable_categorical=True
        )

    model.fit(X_train_final, y_encoded)
    y_test_pred = model.predict(X_test_final)

    return y_test_pred


In [218]:
def preprocess_test_data(df_test, df_train, desc_cols):
    df_test = df_test.copy()
    
    # --- 1. Handle job_title frequency bucketing ---
    job_counts = df_train['job_title'].value_counts()

    def categorize_job(job):
        count = job_counts.get(job, 0)
        if count < 25:
            return 'rare2'
        elif 25 <= count <= 55:
            return 'rare1'
        else:
            return job

    df_test['job_title'] = df_test['job_title'].apply(categorize_job)

    # --- 2. Handle job_state frequency bucketing ---
    state_counts = df_train['job_state'].value_counts()
    states_to_keep = state_counts[state_counts >= 30].index

    df_test['job_state'] = df_test['job_state'].apply(
        lambda x: x if pd.isna(x) or x in states_to_keep else 'other'
    )

    # --- 3. Convert categoricals to match training ---
    df_test['job_title'] = df_test['job_title'].astype('category')
    df_test['job_state'] = df_test['job_state'].astype('category')


    # --- 5. Convert all categorical/objects to numeric codes ---
    for col in df_test.select_dtypes(include=["category", "object"]).columns:
        df_test[col] = df_test[col].astype("category").cat.codes

    # --- 6. Handle any missing columns (ensure test and train align) ---
    expected_columns = [col for col in df_train.columns if col not in ['salary_category', 'salary_cat_num', 'obs']]
    missing_cols = set(expected_columns) - set(df_test.columns)
    for col in missing_cols:
        df_test[col] = 0  # fill missing with 0s

    # --- 7. Ensure column order matches ---
    df_test = df_test[expected_columns]

    return df_test

df_raw_test =preprocess_data("test.csv")
df_test_final = preprocess_test_data(df_raw_test, df_raw, desc_cols)



In [219]:
y_test_pred = predict_on_test(
    best_params=best_params, 
    X_train=X, 
    y_train=y, 
    df_raw_test=df_test_final, 
    desc_cols=desc_cols, 
    model_type='classifier'
)

# Save predictions
df_raw_test["prediction"] = y_test_pred
submission = pd.DataFrame({
    'obs': df_raw_test['obs'],
    'salary_category': le.inverse_transform(y_test_pred)
})


In [220]:
# Define the mapping
category_map = {0: 'Low', 1: 'Medium', 2: 'High'}

# Apply the mapping to  predictions
submission['salary_category'] = submission['salary_category'].map(category_map)


In [221]:
submission.to_csv("res_lda2.csv", index=False)